# Training Loop and Loss Functions

> The previous section covered MoE — a large model is more than a stack of Transformer Blocks; it can also involve routers, experts, and auxiliary losses that complicate training. This section shifts perspective: we follow a single sample as it enters an industrial training library such as Hugging Face Transformers or ModelScope ms-swift, and trace it all the way to a loss value and a parameter update.
>
> We will first hand-compute the loss with tiny numbers, then build a mini Trainer, and finally map each step to the corresponding code in real industrial libraries.

Writing `trainer.train()` in an industrial training library kicks off training. Behind that one line, something like the following happens:

```text
raw text / conversational data
  ↓
Tokenizer and Chat Template
  ↓
input_ids / attention_mask / labels
  ↓
Data Collator assembles a batch
  ↓
model(**batch) produces logits and loss
  ↓
loss.backward()
  ↓
optimizer.step()
  ↓
lr_scheduler.step()
  ↓
save checkpoint / log / evaluate
```

This chapter walks the entire pipeline end to end.

A quick note on the libraries referenced throughout:

| Ecosystem | Common training library | Common entry points |
|:---|:---|:---|
| International | Hugging Face Transformers | `Trainer`, `TrainingArguments`, `DataCollatorForLanguageModeling` |
| China / ModelScope | ModelScope SWIFT, aka `ms-swift` | `swift sft`, `SftArguments`, LoRA/QLoRA/DeepSpeed wrappers |

`ms-swift` is the large-model training toolbox in the ModelScope ecosystem. It covers model loading, datasets, Chat Templates, SFT, LoRA, distributed training, evaluation, and deployment. It does not change how Transformers work; it packages the engineering details of the training loop.


In [ ]:
# All imports for this chapter live in the first code cell
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
print("This chapter uses PyTorch to simulate the core logic behind industrial training libraries.")
print("Key observation: Trainer is not magic — it just wires together data processing, forward, loss, backward, and step.")


## 0. Build the intuition first: what does a Trainer actually save you from?

Let us define three terms.

A Trainer is a training manager. It is not the model itself; it is the program that repeatedly executes training steps. Hugging Face `Trainer` and the SFT entry point of `ms-swift` both fall into this category.

A Batch is a small group of samples fed into the model at once. For example, if one GPU processes 2 conversations at a time, each padded to 8 tokens, the `input_ids` in the batch has shape `[2, 8]`.

Loss is how wrong the model's guess is. The most common loss for language models is Cross-Entropy Loss: the lower the probability assigned to the correct token, the larger the loss.

Zooming out, a training loop is just a sequence of steps executed in order:

```text
1. Pull a batch of raw samples from the dataset
2. tokenizer splits text into token ids
3. data collator pads multiple samples into one batch
4. model forward, produce logits
5. compute cross-entropy loss against labels
6. loss.backward() to compute gradients
7. optimizer.step() to update parameters
8. log, evaluate, save checkpoint
```

In raw PyTorch, the first 7 steps fit in about 10 lines. Industrial training libraries have hundreds of lines because real training also needs to handle memory, mixed precision, gradient accumulation, distribution, resumption, evaluation, checkpointing, and LoRA.

Mathematically, the training loop is always the same sentence: show the model input_ids, predict labels, backprop the loss, update parameters.


## 1. How does a single text sample become a training sample?

We will not use a real tokenizer yet. Instead, hand-build a tiny vocabulary so every number's meaning is obvious.

```text
vocabulary:
0=<pad>, 1=<bos>, 2=<eos>, 3=I, 4=love, 5=machine, 6=learning

original sentence: I love machine learning
full token sequence: [<bos>, I, love, machine, learning, <eos>]
numeric form:       [1,     3,  4,    5,       6,        2]
```

Language-model training does not ask a human to label separate "answers". The answer is the next token in the same sentence:

```text
input_ids = [1, 3, 4, 5, 6]
labels    = [3, 4, 5, 6, 2]
```

That is:

```text
see <bos>                  -> should predict I
see <bos> I                -> should predict love
see <bos> I love           -> should predict machine
see <bos> I love machine   -> should predict learning
see <bos> I love machine learning -> should predict <eos>
```

This is called **next-token prediction**: given the preceding tokens, predict the next token.


In [ ]:
# Build input_ids and labels with concrete numbers
id_to_token = {
    0: "<pad>",
    1: "<bos>",
    2: "<eos>",
    3: "I",
    4: "love",
    5: "machine",
    6: "learning",
}

sentence = torch.tensor([1, 3, 4, 5, 6, 2])
input_ids = sentence[:-1]
labels = sentence[1:]

print("full sentence:", sentence.tolist())
print("input_ids:", input_ids.tolist())
print("labels:   ", labels.tolist())
print()

for pos in range(len(input_ids)):
    context = [id_to_token[i.item()] for i in input_ids[:pos + 1]]
    target = id_to_token[labels[pos].item()]
    print(f"position {pos}: see {context} -> predict {target}")

print()
print("Key observation: labels are not separate human-written answers — they come from shifting the same sentence left/right.")


## 2. Why does the Chat Template affect the loss?

Plain pretraining data is one continuous text, and every token contributes to the loss. But SFT (supervised fine-tuning) often uses conversational data:

```text
user: what is 2 + 2?
assistant: 4
```

Now there is a key question: what should the model learn?

Usually we want the model to learn **how the assistant replies**, not to be penalized for what the user asked. So industrial training libraries build labels like this:

```text
input_ids: [<user>, 2, +, 2, ?, <assistant>, 4, <eos>]
labels:    [-100,  -100, -100, -100, -100, -100, 4, <eos>]
```

The `-100` here is the default ignore marker in PyTorch `cross_entropy`. It means: this position does not contribute to the loss.

**Chat Template**: a template that turns structured dialogue into the text format the model reads, for example by adding special markers around `user` and `assistant`. Different models may use different templates — Qwen, LLaMA, and ChatGLM all have their own formats.

So in `ms-swift` or Transformers, SFT data processing is more than "tokenize":

```text
messages -> apply_chat_template -> tokenize -> labels mask
```


In [ ]:
# Manually simulate one Chat Template and label mask
vocab = {
    "<pad>": 0,
    "<user>": 1,
    "<assistant>": 2,
    "<eos>": 3,
    "2": 4,
    "+": 5,
    "=": 6,
    "?": 7,
    "4": 8,
}

chat_tokens = ["<user>", "2", "+", "2", "?", "<assistant>", "4", "<eos>"]
input_ids = torch.tensor([vocab[token] for token in chat_tokens])

# Train only on the assistant reply: positions 6 and 7 count toward loss, the rest are ignored
labels = torch.tensor([-100, -100, -100, -100, -100, -100, vocab["4"], vocab["<eos>"]])

print("tokens:   ", chat_tokens)
print("input_ids:", input_ids.tolist())
print("labels:   ", labels.tolist())
print()

for token, label in zip(chat_tokens, labels.tolist()):
    if label == -100:
        print(f"{token:>11s} -> no loss")
    else:
        print(f"{token:>11s} -> counts, target token id = {label}")

print()
print("Key observation: SFT does not ask the model to parrot the user; it mainly penalizes the assistant reply.")


## 3. Hand-compute Cross-Entropy: how is the loss actually computed?

At each position the model outputs not a single token but a row of scores called **logits**.

**Logits**: the raw scores the model assigns to every token in the vocabulary, before becoming probabilities. For a vocabulary of 4 tokens, the logits at some position might be:

```text
[2.0, 1.0, 0.1, -1.0]
```

To compute the loss, first apply softmax:

$$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Then look only at the probability of the correct token $p_{correct}$:

$$loss = -\log(p_{correct})$$

Why the negative log? Intuitively:

```text
correct prob = 1.0  -> -log(1.0)  = 0      great
correct prob = 0.5  -> -log(0.5)  = 0.693  okay
correct prob = 0.01 -> -log(0.01) = 4.605  terrible
```


In [ ]:
# Hand-compute the Cross-Entropy loss at one position
logits = torch.tensor([2.0, 1.0, 0.1, -1.0])
correct_id = 0

exp_values = torch.exp(logits)
probs = exp_values / exp_values.sum()
correct_prob = probs[correct_id].item()
manual_loss = -math.log(correct_prob)

torch_loss = F.cross_entropy(logits.view(1, -1), torch.tensor([correct_id]))

print("logits:", logits.tolist())
print("exp(logits):", [round(x, 4) for x in exp_values.tolist()])
print("softmax probs:", [round(x, 4) for x in probs.tolist()])
print()
print(f"correct token id = {correct_id}")
print(f"correct token prob = {correct_prob:.4f}")
print(f"manual loss = -log({correct_prob:.4f}) = {manual_loss:.4f}")
print(f"PyTorch loss = {torch_loss.item():.4f}")
print()
print("Key observation: cross_entropy = log_softmax + pick the correct class + negate.")


## 4. What happens when a batch goes into the model?

Industrial training libraries do not train one sample at a time; they pack many samples into a batch.

Here is the problem: what if two sentences have different lengths?

```text
sample A: [<bos>, I, love, machine, learning, <eos>]     length 6
sample B: [<bos>, I, love, learning, <eos>]              length 5
```

To fit them in the same tensor, we must pad to the same length:

```text
input_ids:
A: [1, 3, 4, 5, 6]
B: [1, 3, 4, 6, 0]

labels:
A: [3, 4, 5, 6, 2]
B: [3, 4, 6, 2, -100]
```

`attention_mask` tells the model which positions are real tokens and which are padding:

```text
A: [1, 1, 1, 1, 1]
B: [1, 1, 1, 1, 0]
```

**Data Collator**: the component that takes multiple already-tokenized samples and organizes them into a batch. It usually handles padding, builds `attention_mask`, and arranges `labels`.


In [ ]:
# A minimal Data Collator that simulates how industrial libraries assemble a batch
def simple_collate(features, pad_id=0, ignore_index=-100):
    """
    Pad multiple samples into one batch.

    Args:
        features: each sample carries input_ids and labels
        pad_id: padding token id for input_ids
        ignore_index: ignore marker for labels

    Returns:
        a batch dict with input_ids, attention_mask, labels
    """
    max_len = max(len(item["input_ids"]) for item in features)

    batch_input_ids = []
    batch_attention_mask = []
    batch_labels = []

    for item in features:
        input_ids = item["input_ids"]
        labels = item["labels"]
        pad_len = max_len - len(input_ids)

        batch_input_ids.append(input_ids + [pad_id] * pad_len)
        batch_attention_mask.append([1] * len(input_ids) + [0] * pad_len)
        batch_labels.append(labels + [ignore_index] * pad_len)

    return {
        "input_ids": torch.tensor(batch_input_ids),
        "attention_mask": torch.tensor(batch_attention_mask),
        "labels": torch.tensor(batch_labels),
    }

features = [
    {"input_ids": [1, 3, 4, 5, 6], "labels": [3, 4, 5, 6, 2]},
    {"input_ids": [1, 3, 4, 6], "labels": [3, 4, 6, 2]},
]

batch = simple_collate(features)

for name, value in batch.items():
    print(f"{name}: shape={tuple(value.shape)}")
    print(value)
    print()

print("Key observation: input_ids is padded with pad_id, labels with -100.")
print("If labels were also padded with pad_id, the model would be forced to learn to predict <pad>, which is wrong.")


## 5. Model forward: why can labels be passed straight into the model?

A common Transformers pattern is:

```python
outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
loss = outputs.loss
logits = outputs.logits
```

This does not mean the model "needs the answer to predict". The real order is:

```text
1. the model runs forward on input_ids and produces logits
2. if you passed labels, the model internally computes the cross_entropy loss for you
3. returns outputs.loss and outputs.logits
```

In other words, `labels` is only used to compute the loss; it never leaks into the model's predictions.

Next we will write a tiny Causal LM. It has no real Transformer Block — just Embedding + Linear to expose the interface. The point is not whether the model is strong, but the shapes and loss computation that a training library relies on.


In [ ]:
class TinyCausalLM(nn.Module):
    """
    A minimal Causal Language Model to demonstrate the interface a Trainer calls.

    Args:
        vocab_size: vocabulary size
        hidden_size: token vector dimension
    """
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_ids, attention_mask=None, labels=None):
        """
        Take token ids, return logits; if labels are provided, also compute loss.

        Args:
            input_ids: [batch, seq_len]
            attention_mask: [batch, seq_len], unused here for simplicity
            labels: [batch, seq_len], positions with -100 do not count toward loss

        Returns:
            a dict with loss and logits
        """
        hidden = self.embedding(input_ids)
        logits = self.lm_head(hidden)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100,
            )

        return {"loss": loss, "logits": logits}

model = TinyCausalLM(vocab_size=9, hidden_size=16)
outputs = model(**batch)

print("logits shape:", tuple(outputs["logits"].shape))
print("loss:", round(outputs["loss"].item(), 4))
print()
print("Key observation: logits is [batch, seq_len, vocab_size].")
print("loss is the mean over all non -100 positions, flattened together.")


## 6. Build a mini Trainer: the core skeleton of an industrial library

Now we put the pieces together. A minimal training step is:

```text
batch = next(dataloader)
outputs = model(**batch)
loss = outputs.loss
loss.backward()
optimizer.step()
optimizer.zero_grad()
```

But industrial training libraries do several more things:

| Engineering feature | Why it is needed |
|:---|:---|
| gradient accumulation | GPU memory cannot fit a large batch, so accumulate gradients over several small batches |
| lr scheduler | the learning rate cannot stay fixed; usually warm up then decay |
| mixed precision | use FP16/BF16 to speed up and save memory |
| gradient clipping | prevent sudden gradient explosions |
| checkpoint | resume after interruptions |
| eval / logging | know whether the model is actually improving |
| distributed training | parallel training across multiple GPUs / machines |

We will first write a mini version without distribution or mixed precision. It is essentially the same as the core of a `Trainer`.


In [ ]:
class ToyTextDataset(Dataset):
    """
    A tiny text dataset where each sample is already token ids.

    Args:
        sequences: multiple full token sequences, each including BOS/EOS
    """
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, index):
        ids = self.sequences[index]
        return {
            "input_ids": ids[:-1],
            "labels": ids[1:],
        }

sequences = [
    [1, 3, 4, 5, 6, 2],
    [1, 3, 4, 6, 2],
    [1, 3, 4, 5, 2],
    [1, 3, 4, 6, 2],
]

dataset = ToyTextDataset(sequences)
dataloader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=simple_collate)

model = TinyCausalLM(vocab_size=9, hidden_size=32)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.05)

print("Before training, look at one batch:")
first_batch = next(iter(dataloader))
for name, value in first_batch.items():
    print(name, tuple(value.shape))
print()
print("Key observation: Dataset handles a single sample; DataLoader + collator assemble the batch.")


In [ ]:
# Mini training loop — this is the core of what a Trainer does
num_epochs = 8
log_every = 1

for epoch in range(num_epochs):
    total_loss = 0.0
    num_steps = 0

    for batch in dataloader:
        outputs = model(**batch)
        loss = outputs["loss"]

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        num_steps += 1

    avg_loss = total_loss / num_steps
    if (epoch + 1) % log_every == 0:
        print(f"epoch {epoch + 1:02d} | train_loss = {avg_loss:.4f}")

print()
print("Key observation: loss drops not because Trainer has magic, but because the optimizer nudges the parameters step by step.")


## 7. Map the mini Trainer onto Hugging Face Transformers

With Transformers, the code looks roughly like this:

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM, not a masked LM like BERT
)

args = TrainingArguments(
    output_dir="outputs/qwen-sft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=500,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=collator,
)

trainer.train()
```

The correspondence with our hand-written loop is:

| Our hand-written component | Transformers component |
|:---|:---|
| `ToyTextDataset` | `Dataset` / `datasets.Dataset` |
| `simple_collate` | `DataCollatorForLanguageModeling` or a custom collator |
| `TinyCausalLM` | `AutoModelForCausalLM` |
| `optimizer` | AdamW created by Trainer, or your custom optimizer |
| `for epoch... for batch...` | `trainer.train()` |
| `outputs = model(**batch)` | Trainer's internal `training_step` |
| `loss.backward()` | Trainer / Accelerate handles backward |
| `optimizer.step()` | Trainer paces the step by gradient accumulation |

One detail to note: the collator for plain-text continued pretraining (CPT) and for conversational SFT is not exactly the same.

| Training type | How labels are usually computed |
|:---|:---|
| Continued pretraining / Causal LM | almost every non-PAD token counts toward loss |
| Instruction tuning / SFT | user/system positions are usually set to `-100`; mainly the assistant reply is trained |


## 8. Map the same pipeline onto ModelScope ms-swift

The "ms-xxx" you mentioned usually refers to **ModelScope SWIFT / `ms-swift`**.

Roughly, it positions itself as a training toolbox for large models and multimodal models. You do not need to write `Trainer`, LoRA injection, template handling, DeepSpeed config, or eval scripts yourself — these pieces are integrated for you.

A typical command looks like:

```bash
swift sft \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --dataset your_dataset.jsonl \
  --train_type lora \
  --learning_rate 2e-5 \
  --per_device_train_batch_size 2 \
  --gradient_accumulation_steps 8 \
  --num_train_epochs 3 \
  --output_dir output/qwen-lora
```

The concepts behind this command are still the loop we just wrote:

| `ms-swift` parameter / feature | Training stage behind it |
|:---|:---|
| `--model` | load the model and tokenizer |
| `--dataset` | read raw samples and turn them into a training dataset |
| model template | turn messages into the model's Chat Template |
| `--train_type lora` | only enable gradients for a subset of low-rank parameters |
| `--per_device_train_batch_size` | how many samples one GPU processes at a time |
| `--gradient_accumulation_steps` | how many small batches make one big update |
| `--learning_rate` | optimizer step size |
| `--deepspeed` / distributed params | shard model states, gradients, optimizer state across GPUs |
| `--output_dir` | where checkpoints and logs go |

It is widely used in the China region because it is tightly integrated with ModelScope models and datasets, the Qwen family, the domestic network environment, and common training configs.

But remember: `ms-swift` is not a different "training principle". It is engineering packaging. What actually happens is still:

```text
batch -> model forward -> loss -> backward -> optimizer step
```


## 9. The full industrial training loop in one diagram

Compress this chapter into one picture:

```text
1. Raw data
   plain text: "I love machine learning"
   dialogue: [{role: user, content: ...}, {role: assistant, content: ...}]

2. Templating
   apply_chat_template(messages)
   -> "<user>...<assistant>..."

3. Tokenize
   tokenizer(text)
   -> input_ids

4. Build labels
   continued pretraining: labels ~= next token of input_ids
   SFT: user/system positions set to -100, assistant positions keep the answer

5. Data Collator
   pad multiple samples to the same length
   -> input_ids / attention_mask / labels

6. Forward
   outputs = model(**batch)
   -> logits: [batch, seq_len, vocab_size]
   -> loss: mean cross-entropy over valid tokens

7. Backward
   loss.backward()
   -> every trainable parameter gets a gradient

8. Optimizer Step
   AdamW / 8-bit Adam / other optimizer
   -> update parameters

9. Scheduler / Logging / Eval / Checkpoint
   adjust learning rate, log loss, run validation, save weights
```

If you use LoRA, add one line:

```text
Most of the original model is frozen; only the small LoRA matrices A/B are trained.
```

If you use DeepSpeed ZeRO / FSDP, add one line:

```text
Parameters, gradients, and optimizer state are sharded across GPUs, but the math behind loss/backward/step does not change.
```


## 10. An information-theoretic view of loss: perplexity, entropy, and KL

When we hand-computed CE earlier, we only cared about "how low is the probability of the correct token". But papers and training logs mention a few more terms: perplexity (PPL), entropy, and KL divergence. They belong to the same family as CE and are often used interchangeably in practice.

This section ties them together: first how PPL converts from CE, then the identity linking CE / entropy / KL, and finally where these identities reappear in distillation, RLHF, DPO, and MoE load balancing.


### 10.1 Perplexity: translate the loss into "how many tokens the model is hesitating among"

Cross-Entropy loss is a number on a log scale, and the raw value is hard to build intuition around. Perplexity (PPL) is the exponentiated version of CE:

$$ \text{PPL} = \exp(\text{CE}) $$

The physical meaning of PPL: at each position, it is equivalent to the model hesitating uniformly among that many tokens.

A few concrete numbers for intuition:

| CE loss | PPL | meaning |
|:---|:---|:---|
| 0 | 1.0 | the model is completely certain of the correct token, no hesitation |
| 2.0 | 7.39 | on average the model hesitates among 7-8 tokens per position |
| 4.6 | 100 | the hesitation range widens to about 100 tokens |
| $\log V$ | $V$ | the model gives a uniform distribution over the vocabulary, equivalent to guessing blindly |

Here $V$ is the vocabulary size. PPL = vocabulary size means the model predicts no better than random guessing, which is the upper bound of PPL.

A common pitfall: PPL is usually not comparable across datasets. Two corpora differ in vocabulary, token boundaries, and domain difficulty, so the absolute CE differs; comparing PPL numbers directly leads to wrong conclusions. When a paper reports PPL, it usually only compares models trained on the same dataset.


In [ ]:
# See the conversion between CE and PPL with concrete numbers
import math

ce_values = [0.0, 1.0, 2.0, 3.0, 4.6, math.log(50000)]

print(f"{'CE':>6} | {'PPL':>12} | intuition")
print("-" * 50)
for ce in ce_values:
    ppl = math.exp(ce)
    if ce == 0.0:
        note = "completely certain, no hesitation"
    elif ce == math.log(50000):
        note = "equivalent to blind guessing on a 50000-token vocab"
    else:
        note = f"on average hesitating among ~{ppl:.1f} tokens"
    print(f"{ce:>6.2f} | {ppl:>12.2f} | {note}")

print()
print("Key observation: PPL translates the log-scale loss onto a linear scale, closer to intuition.")
print("But PPL is usually not comparable across datasets — it must be compared under the same corpus and the same tokenizer.")


### 10.2 The identity linking CE, KL, and entropy

Cross-Entropy, KL divergence, and entropy share one identity:

$$ H(p, q) = H(p) + D_{KL}(p \| q) $$

where $p$ is the true distribution and $q$ is the model's predicted distribution. $H(p, q)$ is cross-entropy, $H(p)$ is the entropy of the true distribution, and $D_{KL}(p \| q)$ is the KL divergence measuring the difference between the two distributions.

Under teacher-forced language-model training, the "true distribution" is one-hot: the correct token has probability 1, all others 0. The entropy of a one-hot distribution is 0:

$$ H(p) = -\sum_i p_i \log p_i = -1 \cdot \log 1 = 0 $$

So under teacher forcing:

$$ \text{CE}(p, q) = D_{KL}(p \| q) $$

This is why, when training a language model, the CE loss and the KL divergence have exactly the same value. It is not a coincidence — it is what happens when one-hot supervision zeroes out the entropy term.



In [ ]:
# Numerically verify: under teacher forcing, CE == KL
import torch
import torch.nn.functional as F

# Vocabulary size 5, correct token is id=2
p = torch.tensor([0.0, 0.0, 1.0, 0.0, 0.0])  # one-hot true distribution
logits = torch.tensor([1.2, 0.5, 2.8, -0.3, 0.1])
q = F.softmax(logits, dim=-1)  # model's predicted distribution

# Cross-Entropy: -sum p_i * log q_i
log_q = torch.log(q)
ce = -(p * log_q).sum().item()

# entropy of p: -sum p_i * log p_i
# only id=2 has p=1, log(1)=0, so H(p) = 0
entropy_p = -(p * torch.log(p.clamp_min(1e-12))).sum().item()

# KL(p || q) = -sum p_i * log(q_i / p_i)
kl = (p * (torch.log(p.clamp_min(1e-12)) - log_q)).sum().item()

print(f"H(p)      = {entropy_p:.4f}")
print(f"CE(p, q)  = {ce:.4f}")
print(f"KL(p||q)  = {kl:.4f}")
print()
print(f"Verify CE == H(p) + KL: {entropy_p:.4f} + {kl:.4f} = {entropy_p + kl:.4f}")
print("Key observation: under teacher forcing the true distribution is one-hot, H(p)=0, so CE and KL coincide.")


### 10.3 Where these identities get reused

CE, KL, and entropy are not only tools for the pretraining loss. They reappear across training paradigms, just under different names and uses.

| Training paradigm | Form used | Why this form |
|:---|:---|:---|
| Pretraining / SFT | CE loss | supervision is one-hot, CE = KL, simple and direct |
| Knowledge distillation | KL divergence | the teacher is a soft-label distribution, no longer one-hot; we need to compare two full distributions |
| RLHF (PPO) | KL penalty | add a KL(policy \| reference) term on top of reward, to keep the policy from drifting |
| DPO | log-ratio (implicit KL) | via log σ(β log(π/π_ref)), equivalent to an implicit KL constraint |
| MoE load balancing | entropy / KL form | encourage the router to spread tokens evenly across experts, preventing a few experts from being overloaded |

MoE load-balancing loss is a typical example. It treats each token's routing probabilities over all experts as a distribution $r$, and wants $r$ to be close to uniform across the batch. Some implementations use an entropy term (maximize the entropy of the router distribution); some write it as KL($r$ \| uniform). At heart they are the same family of operations.

Remember this thread: whenever the training objective involves "pushing one distribution toward another", the trio of CE / KL / entropy shows up. Teacher forcing is the most special case — one-hot supervision makes all three numerically equal, which is why the pretraining loss can be written so concisely as CE.


## 11. A confusing point: does the shift happen inside the model or in the data?

Some tutorials write:

```python
input_ids = tokens[:-1]
labels = tokens[1:]
```

But many Transformers models, when training, write:

```python
input_ids = tokens
labels = tokens
```

This is not a contradiction. The difference is **where the shift happens**.

| Form | Where the shift happens | Good for teaching |
|:---|:---|:---|
| `input=tokens[:-1]`, `labels=tokens[1:]` | at the data-processing stage | best for teaching, the relationship is clearest |
| `input=tokens`, `labels=tokens` | inside the model when computing loss | common in industrial libraries, skips manual slicing |

Many `AutoModelForCausalLM` models internally align logits and labels when computing loss:

```text
shift_logits = logits[..., :-1, :]
shift_labels = labels[..., 1:]
```

So when you see `labels=input_ids`, do not misread it as "the model predicts itself". What is actually compared is still: predict the next token from the current position.


In [ ]:
# Compare: external shift in the data vs internal shift in the model — same supervision
full_tokens = torch.tensor([[1, 3, 4, 5, 6, 2]])

external_input = full_tokens[:, :-1]
external_labels = full_tokens[:, 1:]

internal_input = full_tokens
internal_labels = full_tokens
shifted_labels_inside_model = internal_labels[:, 1:]

print("External shift:")
print("input: ", external_input.tolist())
print("labels:", external_labels.tolist())
print()

print("Equivalent result of internal shift:")
print("input passes full tokens:    ", internal_input.tolist())
print("model takes labels[:, 1:]:", shifted_labels_inside_model.tolist())
print()

same = torch.equal(external_labels, shifted_labels_inside_model)
print("Do both forms give the same prediction target?", same)
print("Key observation: industrial libraries often hide the shift inside the model's loss computation.")


## Homework

> You may ask AI to explain the ideas, break down the steps, or check your direction. But do not let AI "do the whole problem for you".


**Homework 1: build SFT labels**

Given a conversation that has already been templated into tokens:

```text
tokens = ["<user>", "hello", "<assistant>", "hi", "there", "<eos>"]
ids    = [1,       10,   2,            10,   11,   3]
```

Build `labels` so that only the assistant reply `["hi", "there", "<eos>"]` contributes to loss; set the rest to `-100`.

Hint: the assistant reply starts at id 10 and ends at EOS id 3.


In [ ]:
# Homework 1: build SFT labels
ids = [1, 10, 2, 10, 11, 3]

# TODO: keep only the assistant reply part as labels, set the rest to -100
labels = None  # fill in here, e.g. [-100, ...]

assert labels is not None, "build labels first"
assert labels == [-100, -100, -100, 10, 11, 3], f"labels are wrong, you got {labels}"

print("✅ Homework 1 passed:")
print("   You can mask out the user/template parts and train only on the assistant reply.")


**Homework 2: compute the effective batch size under gradient accumulation**

Training parameters:

```text
per_device_train_batch_size = 2
num_gpus = 4
gradient_accumulation_steps = 8
```

Compute how many samples one optimizer step effectively sees.

Hint: effective batch size = per-GPU batch × number of GPUs × gradient accumulation steps.


In [ ]:
# Homework 2: compute the effective batch size
per_device_train_batch_size = 2
num_gpus = 4
gradient_accumulation_steps = 8

# TODO: compute how many samples accumulate before one optimizer.step()
effective_batch_size = None

assert effective_batch_size is not None, "compute effective_batch_size first"
assert effective_batch_size == 64, f"expected 64, you got {effective_batch_size}"

print("✅ Homework 2 passed:")
print(f"   effective batch size = {effective_batch_size}")
print("   This is why gradient accumulation is common when GPU memory is tight.")


**Homework 3: average the loss only over valid tokens**

A flattened batch has 6 positions, each with a loss:

```text
losses = [0.2, 0.4, 1.0, 0.3, 0.8, 0.5]
labels = [10, 11, -100, 3, -100, 12]
```

Compute the average loss over valid positions. Positions with `labels == -100` must be ignored.

Hint: the valid positions are 0, 1, 3, 5.


In [ ]:
# Homework 3: average loss only over valid tokens
losses = [0.2, 0.4, 1.0, 0.3, 0.8, 0.5]
labels = [10, 11, -100, 3, -100, 12]

# TODO: filter out positions where labels == -100, then take the mean
valid_average_loss = None

assert valid_average_loss is not None, "compute valid_average_loss first"
expected = (0.2 + 0.4 + 0.3 + 0.5) / 4
assert abs(valid_average_loss - expected) < 1e-6, f"expected {expected:.4f}"

print("✅ Homework 3 passed:")
print(f"   valid average loss = {valid_average_loss:.4f}")
print("   You now understand the core role of ignore_index=-100.")


## Summary (checklist)

Confirm you understand these points:

1. ✅ `Trainer` / `ms-swift` is not magic; it is engineering packaging around the training loop
2. ✅ A sample goes through: templating → tokenize → build labels → collator assembles batch → model forward → loss
3. ✅ The core objective of a Causal LM is still next-token prediction
4. ✅ `logits` scores the whole vocabulary at each position; Cross-Entropy only penalizes the correct token for not being probable enough
5. ✅ `-100` means this position does not contribute to loss; commonly used for PAD and for user/system parts in SFT
6. ✅ `attention_mask` controls whether the model "looks at" pad positions; `labels=-100` controls whether the loss "counts" them
7. ✅ Hugging Face Transformers `Trainer` and ModelScope `ms-swift` both add engineering on top of this same pipeline
8. ✅ `gradient_accumulation_steps` changes how often parameters are updated, not the per-forward batch shape
9. ✅ LoRA only changes "which parameters are trainable"; it does not change the basic logic of loss/backward/optimizer step
10. ✅ DeepSpeed/FSDP change how parameters and optimizer state are distributed across GPUs, not the mathematical objective of training

One-line summary: industrial training libraries wrap a lot of engineering detail, but as long as you hold on to the main thread `batch -> model -> loss -> backward -> step`, `trainer.train()` will not intimidate you.

The next section moves to Scaling Laws: once you know how to train, the next question is "how big should the model be, how much data, how much compute, to be worth it".
